# 10.07 - Optimization for CV

**Notebook type:** Solution notebook with full working code and test cases.

**Daily output:** a CV optimization ablation comparing three learning-rate and weight-decay settings on the same small CNN.

Today is about controlling training instead of only making a model bigger: optimizer choice, learning rate, scheduler behavior, weight decay, and reading train/validation curves for overfitting signals.


## Core Ideas

Optimization choices decide how a CNN moves through the loss landscape.

- **Learning rate:** the step size. Too small can underfit slowly; too large can bounce or diverge.
- **Optimizer:** SGD with momentum is simple and strong; Adam adapts per-parameter step sizes and often learns quickly.
- **Weight decay:** penalizes large weights and can reduce overfitting.
- **Scheduler:** changes learning rate over epochs, often lowering it after the model reaches a plateau.
- **Overfitting curves:** training loss keeps improving while validation loss stops improving or gets worse.

For a fair ablation, keep the dataset, architecture, batch size, seed, and number of epochs fixed. Change only the optimizer settings you are testing.


In [ ]:
import random
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed. Complete this notebook in an environment with torch.")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)

set_seed(SEED)
if TORCH_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
else:
    device = None


## Prepared Image Data

Run this cell before the exercises. The synthetic optimization dataset is provided so the ablation work can focus on DataLoaders, the CNN, optimizer settings, and curve analysis.


In [ ]:
def make_optimization_dataset(n_per_class=60, image_size=16, noise=0.14, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    generator = torch.Generator().manual_seed(seed)
    images = []
    labels = []

    for class_id in range(3):
        for _ in range(n_per_class):
            img = torch.zeros(1, image_size, image_size, dtype=torch.float32)
            center = image_size // 2
            if class_id == 0:
                img[:, :, center - 2:center] = 1.0
                img[:, :, center + 3:center + 4] = 0.65
            elif class_id == 1:
                img[:, center - 2:center, :] = 1.0
                img[:, center + 3:center + 4, :] = 0.65
            else:
                for i in range(image_size):
                    img[:, i, i] = 1.0
                    if i + 1 < image_size:
                        img[:, i, i + 1] = 0.75

            img = img + noise * torch.randn(img.shape, generator=generator)
            images.append(img.clamp(0.0, 1.0))
            labels.append(class_id)

    X = torch.stack(images)
    y = torch.tensor(labels, dtype=torch.long)
    perm = torch.randperm(len(y), generator=generator)
    return X[perm], y[perm]


if TORCH_AVAILABLE:
    X, y = make_optimization_dataset()
    print("X:", X.shape, X.dtype)
    print("y:", y.shape, y.dtype, sorted(y.unique().tolist()))


## Exercise 10-A: DataLoaders and CNN Setup

Use the prepared tensors `X` and `y`. Build deterministic train/validation loaders and a compact CNN that outputs logits shaped `[batch, num_classes]`.


In [ ]:
def build_loaders(X, y, batch_size=32, train_frac=0.75, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    n = len(y)
    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=generator)
    train_n = int(train_frac * n)
    train_idx = perm[:train_n]
    val_idx = perm[train_n:]

    train_ds = TensorDataset(X[train_idx], y[train_idx])
    val_ds = TensorDataset(X[val_idx], y[val_idx])
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


class TinyOptimizationCNN(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, num_classes=3):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


if TORCH_AVAILABLE:
    train_loader, val_loader = build_loaders(X, y)
    xb, yb = next(iter(train_loader))
    model = TinyOptimizationCNN().to(device)
    with torch.no_grad():
        logits = model(xb.to(device))
    print("batch:", xb.shape, yb.shape)
    print("logits:", logits.shape)


## Exercise 10-B: Optimizer and Scheduler Builder

Write a config-driven optimizer builder. It should support at least `adam` and `sgd`, learning rate, weight decay, SGD momentum, and optional schedulers.

Recommended scheduler options:

- `none`: keep LR constant
- `step`: multiply LR by `gamma` every `step_size` epochs
- `cosine`: cosine decay over the planned number of epochs


In [ ]:
def build_optimizer_and_scheduler(model, config):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    optimizer_name = config.get("optimizer", "adam").lower()
    lr = float(config.get("lr", 1e-3))
    weight_decay = float(config.get("weight_decay", 0.0))

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        momentum = float(config.get("momentum", 0.9))
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    scheduler_name = config.get("scheduler", "none").lower()
    if scheduler_name in {"none", ""}:
        scheduler = None
    elif scheduler_name == "step":
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=int(config.get("step_size", 2)),
            gamma=float(config.get("gamma", 0.5)),
        )
    elif scheduler_name == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=int(config.get("t_max", config.get("epochs", 4))),
        )
    else:
        raise ValueError(f"Unsupported scheduler: {scheduler_name}")

    return optimizer, scheduler


def get_current_lr(optimizer):
    return float(optimizer.param_groups[0]["lr"])


if TORCH_AVAILABLE:
    demo_model = TinyOptimizationCNN().to(device)
    demo_config = {
        "optimizer": "sgd",
        "lr": 0.05,
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "scheduler": "step",
        "step_size": 2,
        "gamma": 0.5,
    }
    demo_optimizer, demo_scheduler = build_optimizer_and_scheduler(demo_model, demo_config)
    print(type(demo_optimizer).__name__, "lr:", get_current_lr(demo_optimizer), "scheduler:", type(demo_scheduler).__name__)


## Exercise 10-C: Train One Experiment

Implement the reusable training functions. Each epoch should record:

- `epoch`
- `train_loss`
- `train_acc`
- `val_loss`
- `val_acc`
- `lr`

Call `model.train()` during training and `model.eval()` plus `torch.no_grad()` during validation.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_seen += len(yb)

    return total_loss / total_seen, total_correct / total_seen


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_seen += len(yb)

    return {"loss": total_loss / total_seen, "accuracy": total_correct / total_seen}


def run_training_experiment(config, train_loader, val_loader, epochs=4, seed=42, device=device):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    set_seed(seed)
    model = TinyOptimizationCNN(num_classes=int(config.get("num_classes", 3))).to(device)
    criterion = nn.CrossEntropyLoss()
    config = dict(config)
    config.setdefault("epochs", epochs)
    optimizer, scheduler = build_optimizer_and_scheduler(model, config)

    history = []
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "lr": get_current_lr(optimizer),
        }
        history.append(row)
        if scheduler is not None:
            scheduler.step()

    best_row = min(history, key=lambda row: row["val_loss"])
    best_acc = max(row["val_acc"] for row in history)
    return {
        "name": config.get("name", "unnamed"),
        "config": config,
        "history": history,
        "best_val_loss": best_row["val_loss"],
        "best_val_acc": best_acc,
        "final_val_loss": history[-1]["val_loss"],
        "final_val_acc": history[-1]["val_acc"],
    }


if TORCH_AVAILABLE:
    demo_result = run_training_experiment(
        {"name": "demo_adam", "optimizer": "adam", "lr": 0.003, "weight_decay": 1e-4, "scheduler": "none"},
        train_loader,
        val_loader,
        epochs=2,
        seed=SEED,
        device=device,
    )
    print(demo_result["name"], demo_result["history"][-1])


## Exercise 10-D: Run a Three-Setting Ablation

Run three optimizer settings on the same CNN and compare the curves.

Include one deliberately aggressive setting, one Adam setting with weight decay, and one SGD setting with momentum. Store enough information to identify the best validation result.


In [ ]:
ABLATION_CONFIGS = [
    {"name": "adam_lr_1e-2_no_decay", "optimizer": "adam", "lr": 1e-2, "weight_decay": 0.0, "scheduler": "none"},
    {"name": "adam_lr_3e-3_decay_step", "optimizer": "adam", "lr": 3e-3, "weight_decay": 1e-4, "scheduler": "step", "step_size": 2, "gamma": 0.5},
    {"name": "sgd_lr_5e-2_decay_step", "optimizer": "sgd", "lr": 5e-2, "momentum": 0.9, "weight_decay": 1e-4, "scheduler": "step", "step_size": 2, "gamma": 0.5},
]


def run_ablation(configs, train_loader, val_loader, epochs=4, seed=42, device=device):
    results = []
    for config in configs:
        result = run_training_experiment(
            config,
            train_loader,
            val_loader,
            epochs=epochs,
            seed=seed,
            device=device,
        )
        results.append(result)
    return results


def rank_ablation_results(results):
    summary = []
    for result in results:
        history = result["history"]
        summary.append({
            "name": result["name"],
            "optimizer": result["config"].get("optimizer"),
            "lr": result["config"].get("lr"),
            "weight_decay": result["config"].get("weight_decay", 0.0),
            "best_val_loss": result["best_val_loss"],
            "best_val_acc": result["best_val_acc"],
            "final_val_loss": result["final_val_loss"],
            "final_val_acc": result["final_val_acc"],
            "final_lr": history[-1]["lr"],
        })
    return sorted(summary, key=lambda row: row["best_val_loss"])


if TORCH_AVAILABLE:
    ablation_results = run_ablation(ABLATION_CONFIGS, train_loader, val_loader, epochs=4, seed=SEED, device=device)
    for row in rank_ablation_results(ablation_results):
        print(row)


## Exercise 10-E: Diagnose Optimization Curves

Turn the history into a short diagnosis. Look for:

- overfitting: high train accuracy, lower validation accuracy, and validation loss rebound
- underfitting: both train and validation accuracy remain low
- improving: validation loss is moving down by the end
- unstable: validation loss is not clearly improving


In [ ]:
def diagnose_curve(history, gap_threshold=0.15, loss_rebound=0.05):
    if not history:
        raise ValueError("history must contain at least one epoch")

    first = history[0]
    final = history[-1]
    best_val_loss = min(row["val_loss"] for row in history)
    acc_gap = final["train_acc"] - final["val_acc"]
    rebound_amount = final["val_loss"] - best_val_loss

    if acc_gap >= gap_threshold and rebound_amount > loss_rebound:
        label = "overfitting"
    elif final["train_acc"] < 0.70 and final["val_acc"] < 0.70:
        label = "underfitting"
    elif final["val_loss"] < first["val_loss"]:
        label = "improving"
    else:
        label = "unstable"

    return {
        "label": label,
        "acc_gap": acc_gap,
        "best_val_loss": best_val_loss,
        "final_val_loss": final["val_loss"],
    }


def make_ablation_report(results):
    ranked = rank_ablation_results(results)
    diagnoses = {result["name"]: diagnose_curve(result["history"])["label"] for result in results}
    report = []
    for row in ranked:
        report.append({
            "name": row["name"],
            "best_val_loss": row["best_val_loss"],
            "best_val_acc": row["best_val_acc"],
            "final_val_acc": row["final_val_acc"],
            "diagnosis": diagnoses[row["name"]],
        })
    return report


if TORCH_AVAILABLE:
    for row in make_ablation_report(ablation_results):
        print(row)


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 10 tests passed`.


In [ ]:
def run_day10_tests():
    if not TORCH_AVAILABLE:
        print("Day 10 tests skipped because PyTorch is not installed.")
        return

    required_names = [
        "make_optimization_dataset",
        "build_loaders",
        "TinyOptimizationCNN",
        "build_optimizer_and_scheduler",
        "get_current_lr",
        "train_one_epoch",
        "evaluate",
        "run_training_experiment",
        "run_ablation",
        "rank_ablation_results",
        "diagnose_curve",
        "make_ablation_report",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_optimization_dataset(n_per_class=6, image_size=16, seed=123)
    assert X_test.shape == (18, 1, 16, 16), f"Unexpected X shape: {X_test.shape}"
    assert y_test.shape == (18,), f"Unexpected y shape: {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    train_loader_test, val_loader_test = build_loaders(X_test, y_test, batch_size=6, train_frac=0.67, seed=123)
    xb, yb = next(iter(train_loader_test))
    assert xb.ndim == 4 and xb.shape[1:] == (1, 16, 16)
    assert yb.dtype == torch.long

    model_test = TinyOptimizationCNN(num_classes=3).to(device)
    with torch.no_grad():
        logits = model_test(xb.to(device))
    assert logits.shape == (xb.shape[0], 3), f"Unexpected logits shape: {logits.shape}"

    adam_config = {
        "name": "test_adam",
        "optimizer": "adam",
        "lr": 0.003,
        "weight_decay": 1e-4,
        "scheduler": "step",
        "step_size": 1,
        "gamma": 0.5,
    }
    optimizer, scheduler = build_optimizer_and_scheduler(model_test, adam_config)
    assert isinstance(optimizer, torch.optim.Optimizer)
    assert abs(get_current_lr(optimizer) - 0.003) < 1e-12
    if scheduler is not None:
        optimizer.step()
        scheduler.step()
        assert get_current_lr(optimizer) < 0.003

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_test.parameters(), lr=0.003)
    train_loss, train_acc = train_one_epoch(model_test, train_loader_test, criterion, optimizer, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(model_test, val_loader_test, criterion, device)
    assert {"loss", "accuracy"}.issubset(metrics.keys())
    assert isinstance(metrics["loss"], float)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    quick_configs = [
        {"name": "quick_adam", "optimizer": "adam", "lr": 0.003, "weight_decay": 0.0, "scheduler": "none"},
        {"name": "quick_sgd", "optimizer": "sgd", "lr": 0.03, "momentum": 0.9, "weight_decay": 1e-4, "scheduler": "step", "step_size": 1, "gamma": 0.5},
    ]
    results = run_ablation(quick_configs, train_loader_test, val_loader_test, epochs=2, seed=123, device=device)
    assert len(results) == 2
    for result in results:
        assert {"name", "config", "history", "best_val_loss", "best_val_acc", "final_val_loss", "final_val_acc"}.issubset(result.keys())
        assert len(result["history"]) == 2
        assert {"epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"}.issubset(result["history"][0].keys())

    ranked = rank_ablation_results(results)
    assert len(ranked) == 2
    assert ranked[0]["best_val_loss"] <= ranked[-1]["best_val_loss"]

    overfit_history = [
        {"train_loss": 0.50, "train_acc": 0.80, "val_loss": 0.40, "val_acc": 0.78},
        {"train_loss": 0.15, "train_acc": 0.99, "val_loss": 0.70, "val_acc": 0.62},
    ]
    underfit_history = [
        {"train_loss": 1.20, "train_acc": 0.35, "val_loss": 1.25, "val_acc": 0.30},
        {"train_loss": 1.10, "train_acc": 0.45, "val_loss": 1.18, "val_acc": 0.38},
    ]
    assert diagnose_curve(overfit_history)["label"] == "overfitting"
    assert diagnose_curve(underfit_history)["label"] == "underfitting"

    report = make_ablation_report(results)
    assert len(report) == 2
    assert {"name", "best_val_loss", "best_val_acc", "final_val_acc", "diagnosis"}.issubset(report[0].keys())

    print("Day 10 tests passed")

run_day10_tests()


## Day 10 Checklist

Before trusting an optimization ablation, verify that all runs use the same model and data split, each config records learning rate and weight decay, validation is measured with `eval()` and `no_grad()`, the best run is selected by validation metrics, and curve diagnosis is based on both train and validation behavior.
